
## Explode, Pivot, and Complex Types:

In [0]:
%run ./05_Window_Functions


## Explode, Pivot, Complex Types are started here:


## 1. explode():

**explode()** is a PySpark function is used to `converts each element of an array into a separate row in a DataFrame`.

In [0]:
customer_preferences_data = [
    (101, "Arun", ["Electronics", "Accessories"]),
    (102, "Priya", ["Furniture", "Electronics"]),
    (103, "Karthik", ["Accessories", "Electronics"])
]

customer_preferences_df = spark.createDataFrame(
    customer_preferences_data,
    ["customer_id", "customer_name", "preferred_categories"]
)

display(customer_preferences_df)

In [0]:
exploded_preferences_df = customer_preferences_df.select(
    "customer_id",
    "customer_name",
    F.explode("preferred_categories").alias("preferred_category")
)

display(exploded_preferences_df)


**Business use: Analyze how many customers prefer each product category.**


### 2. pivot():

- **pivot()** is used` to transforms distinct values from a column into separate columns`, allowing you to `summarize data by categories.
`
- here we can use aggregation such as sum(), count(), or avg().

In [0]:
monthly_category_sales_df = sales_df.groupBy( "order_month" ).pivot(
    "category",
    ["Electronics", "Accessories", "Furniture"]
).agg(
    F.sum("net_revenue")
)

display(monthly_category_sales_df)

### Complex Types:

- Complex Types are **data types** is used to `store multiple values of same data type or structured fields inside a single DataFrame column`.
- Complex Types such as `ArrayType, MapType, and StructType`.


### 1. Complex Type: Array

It is used to `stores multiple values of the same data type in one column`.

In [0]:
array_df = sales_df.groupBy(
    "customer_id",
    "customer_name"
).agg(
    F.collect_list("product_name").alias("purchased_products")
)

display(array_df)


**Business use: Store the list of products purchased by each customer.**


### Group by customer and product:

In [0]:
customer_product_quantity_df = sales_df.groupBy(
    "customer_id",
    "customer_name",
    "product_name"
).agg(
    F.sum("quantity").alias("total_quantity")
)

display(customer_product_quantity_df)


### 2. Complex Type: Map

It is used to `stores key-value pairs in one column`.

### Find duplicate product keys

In [0]:
from pyspark.sql import functions as F

customer_product_quantity_df.groupBy(
    "customer_id",
    "customer_name",
    "product_name"
).count().filter(
    (F.col("count") > 1) |
    F.col("product_name").isNull()
).display()

### Rebuild the data using a unique customer-product key

In [0]:
customer_product_quantity_df = sales_df.groupBy(
    "customer_id",
    "product_name"
).agg(
    F.sum("quantity").alias("total_quantity"),
    F.first("customer_name", ignorenulls=True).alias("customer_name")
)

display(customer_product_quantity_df)

### Exclude null product names and build the map

In [0]:
customer_map_df = customer_product_quantity_df.filter(
    F.col("product_name").isNotNull()
).groupBy(
    "customer_id",
    "customer_name"
).agg(
    F.map_from_entries(
        F.collect_list(
            F.struct(
                F.col("product_name"),
                F.col("total_quantity")
            )
        )
    ).alias("product_quantity_map")
)

display(customer_map_df)


### 3. Complex Type: Struct

It is used to `stores name fields where one column contains different data types`.

In [0]:
customer_struct_df = sales_df.select(
    "order_id",
    F.struct(
        F.col("customer_id"),
        F.col("customer_name"),
        F.col("city"),
        F.col("membership")
    ).alias("customer_details"),
    F.col("net_revenue")
)

display(customer_struct_df)

In [0]:
customer_struct_df.select(
    "order_id",
    F.col("customer_details.customer_name").alias("customer_name"),
    F.col("customer_details.city").alias("city"),
    "net_revenue"
).display()


**Business use: Organize related customer information into a nested structure for analytics or JSON output.**